In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mdsadiqiqbal/txd-22-a-large-scale-benchmark-dataset/TXD-22 A Large-Scale Benchmark Dataset.csv


In [2]:
import io, os, re, time, copy, json, random, codecs, glob
from collections import Counter
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg") #saves figures to files
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import kagglehub

SEED = 42
MAX_IN, MAX_OUT = 64, 400 #max tokens kept from the input/output
VOCAB_SIZE = 40000
EMBED_DIM = 128 #the CNN tutorial uses 300; 128 is smaller
EPOCHS = 10
BATCH = 50
PATIENCE = 3 #early stopping on validation accuracy
OUT_DIR = "outputs_6"
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

#data
LLMS = ["ChatGPT", "Claude", "Qwen", "Llama", "Gemini", "DeepSeek"] #using 6 classes

def load_data():
    def cp1252_fallback(err): #the CSV mixes UTF-8 and cp1252 so decode as UTF-8, fall back to cp1252 per bad byte
        return err.object[err.start:err.end].decode("cp1252", errors = "replace"), err.end
    codecs.register_error("cp1252_fallback", cp1252_fallback)
    found = glob.glob("/kaggle/input/**/TXD-22*.csv", recursive = True) #the dataset attached to this notebook
    if found:
        csv_path = found[0]
    else: #not attached, so download it
        path = kagglehub.dataset_download("mdsadiqiqbal/txd-22-a-large-scale-benchmark-dataset")
        csv_path = os.path.join(path, "TXD-22 A Large-Scale Benchmark Dataset.csv")
    print("reading:", csv_path)
    raw = open(csv_path, "rb").read()
    return pd.read_csv(io.StringIO(raw.decode("utf-8-sig", errors = "cp1252_fallback")))

def prepare(df):
    data = df[df.Source.isin(LLMS)].copy() #keep only the chosen chatbots
    data = data.drop_duplicates("Answer", keep = False) #removes identical answers
    nsrc = data.groupby("Question").Source.transform("nunique")
    data = data[nsrc >= len(LLMS)].reset_index(drop = True) #keep only questions answered by ALL chatbots in LLMS
    classes = sorted(data.Source.unique())
    data["y"] = data.Source.map({c: i for i, c in enumerate(classes)}) # assigns LLM to a number
    return data, classes

df = load_data()
data, classes = prepare(df)
class_count = len(classes)
print(f"rows: {len(data)} | classes: {class_count} | chance: {1/class_count:.3f}")

#split by question
#70% training, 15% validation, 15% testing
questions = data.Question.unique()
q_train, q_tmp = train_test_split(questions, test_size = 0.30, random_state = SEED)
q_val, q_test = train_test_split(q_tmp, test_size = 0.50, random_state = SEED)
train = data[data.Question.isin(q_train)].copy()
val = data[data.Question.isin(q_val)].copy()
test = data[data.Question.isin(q_test)].copy()
print("train/val/test:", len(train), len(val), len(test))

#tokenizer
TOK = re.compile(r"\n|\w+|[^\w\s]")
def tokenize(s):
    return ["<NL>" if t == "\n" else t for t in TOK.findall(s)]
 
counts = Counter()
for text in pd.concat([train.Question, train.Answer]):
    counts.update(tokenize(text))
index_to_string = ["<pad>", "<unk>", "<sep>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 3)]
word_to_index = {w: i for i, w in enumerate(index_to_string)}
encode = lambda s: [word_to_index.get(t, 1) for t in tokenize(s)]
for part in (train, val, test):
    part["q_ids"] = part.Question.map(encode)
    part["a_ids"] = part.Answer.map(encode)

def make_dataset(part, mode):
    seqs = []
    for question, answer in zip(part.q_ids, part.a_ids):
        if mode == "input": 
            sequence = question[:MAX_IN]
        elif mode == "output": 
            sequence = answer[:MAX_OUT]
        else: 
            sequence = question[:MAX_IN]+ [word_to_index["<sep>"]] + answer[:MAX_OUT]
        seqs.append(sequence if sequence else [1])
    longest = 0
    for i in range(len(seqs)):
        if len(seqs[i]) > longest:
            longest = len(seqs[i])
    max_length = max(5, longest)
    input_matrix = np.zeros((len(seqs), max_length), dtype = np.int64) 
    for i, sequence in enumerate(seqs):
        input_matrix[i, :len(sequence)] = sequence
    sequence_lengths = torch.tensor([len(sequence) for sequence in seqs], dtype = torch.int64)
    return TensorDataset(torch.tensor(input_matrix), sequence_lengths, torch.tensor(part.y.values, dtype = torch.long))

#models
#CNN
class CNN_NLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes = (3, 4, 5), num_filters = (100, 100, 100), dropout = 0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx = 0, max_norm = 5.0)
        self.conv1d_list = nn.ModuleList([
            nn.Conv1d(in_channels = embed_dim, out_channels = num_filter, kernel_size = filter_size)
            for filter_size, num_filter in zip(filter_sizes, num_filters)])
        self.fc = nn.Linear(sum(num_filters), num_classes)
        self.dropout = nn.Dropout(p = dropout)

    def forward(self, input_ids, lengths = None):
        x = self.embedding(input_ids).permute(0, 2, 1) #(b, embed, len) for Conv1d
        x_conv_list = [F.relu(conv(x)) for conv in self.conv1d_list] #(b, nf, len-fs+1)
        x_pool_list = [F.max_pool1d(conv_output, kernel_size = conv_output.shape[2]).squeeze(2) #1-max pooling over time
                  for conv_output in x_conv_list]
        return self.fc(self.dropout(torch.cat(x_pool_list, dim = 1))) #logits

#RNN - LSTM
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, dropout = 0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx = 0)
        self.rnn = nn.LSTM(embed_dim, hidden_size, batch_first = True, bidirectional = True)
        self.h2o = nn.Linear(hidden_size * 2, num_classes) #2 directions
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.LogSoftmax(dim = 1)

    def forward(self, input_ids, lengths):
        emb = self.dropout(self.embedding(input_ids))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True,
                                                   enforce_sorted=False)
        _, (hidden, _) = self.rnn(packed) #hidden: (2, batch, hidden_size)
        combined_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1) 
        return self.softmax(self.h2o(self.dropout(combined_hidden)))

def evaluate_model(model, loader, loss_fn):
    model.eval()
    losses = []
    preds = []
    all_true_labels = []
    with torch.no_grad():
        for input_sequence, lens, true_labels in loader:
            out = model(input_sequence.to(device), lens)
            losses.append(loss_fn(out, true_labels.to(device)).item())
            preds.append(out.argmax(1).cpu())
            all_true_labels.append(true_labels)
    actual_labels, predicted_labels = torch.cat(all_true_labels).numpy(), torch.cat(preds).numpy()
    return float(np.mean(losses)), accuracy_score(actual_labels, predicted_labels), actual_labels, predicted_labels

def train_model(model, optimizer, loss_function, train_loader, validation_loader, epochs, clip = None):
    best_accuracy = -1.0
    best_epoch = 0
    best_state = None
    history = []
    print(f"{'Epoch':^7} | {'Train Loss':^12} | {'Val Loss':^10} | {'Val Accuracy':^9} | {'Elapsed':^9}")
    print("-" * 60)
    for epoch in range(1, epochs + 1):
        initial_time, total = time.time(), 0.0
        model.train()
        for input_sequences, lens, true_labels in train_loader:
            optimizer.zero_grad()
            loss = loss_function(model(input_sequences.to(device), lens), true_labels.to(device))
            loss.backward()
            if clip: 
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            total += loss.item()
        train_loss = total/len(train_loader)
        val_loss, val_accuracy, _, _ = evaluate_model(model, validation_loader, loss_function)
        history.append((train_loss, val_loss, val_accuracy))
        print(f"{epoch:^7} | {train_loss:^12.6f} | {val_loss:^10.6f} | {val_accuracy * 100:^9.2f} | {time.time() - initial_time:^9.2f}")
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= PATIENCE:
            print("early stopping")
            break
    model.load_state_dict(best_state) #test with the best epoch's weights
    return best_accuracy, history
  
def create_confusion_matrix(confusion_matrix, name):
    cmn = confusion_matrix/confusion_matrix.sum(axis = 1, keepdims = True).clip(min = 1)
    fig = plt.figure(figsize = (8, 7))
    ax = fig.add_subplot(111)
    cax = ax.matshow(cmn)
    fig.colorbar(cax)
    ax.set_xticks(range(class_count), labels = classes, rotation = 90)
    ax.set_yticks(range(class_count), labels = classes)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true") 
    ax.xaxis.set_label_position("top")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"cm_{name}.png"), dpi = 120)
    plt.close(fig)

def plot_curves(history, name):
    history_arr = np.array(history)
    fig, ax = plt.subplots(figsize = (5, 3.5))
    ax.plot(history_arr[:, 0], label = "training loss")
    ax.plot(history_arr[:, 1], label = "validation loss")
    ax.set_xlabel("epoch")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"loss_{name}.png"), dpi = 120)
    plt.close(fig)

def run_model(model_architecture, mode):
    set_seed()
    train_loader = DataLoader(make_dataset(train, mode), batch_size = BATCH, shuffle = True)
    validation_loader = DataLoader(make_dataset(val, mode), batch_size = 256)
    test_loader = DataLoader(make_dataset(test, mode), batch_size = 256)
    if model_architecture == "cnn":
        model = CNN_NLP(len(index_to_string), EMBED_DIM, class_count).to(device)
        optimizer = torch.optim.Adadelta(model.parameters(), lr = 0.25, rho = 0.95)
        loss_function, clip = nn.CrossEntropyLoss(), None
    else:
        model= RNNClassifier(len(index_to_string), EMBED_DIM, 128, class_count).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
        loss_function, clip = nn.NLLLoss(), 3
    print(f"\n{model_architecture.upper()} | input type = {mode} | parameters = {sum(p.numel() for p in model.parameters()):,}")
    best_validation, history = train_model(model, optimizer, loss_function, train_loader, validation_loader, EPOCHS, clip)
    _, test_acc, actual_labels, predicted_labels = evaluate_model(model, test_loader, loss_function)
    f1 = f1_score(actual_labels, predicted_labels, average = "macro")
    conf_matrix = confusion_matrix(actual_labels, predicted_labels, labels = range(class_count))
    create_confusion_matrix(conf_matrix, f"{model_architecture}_{mode}")
    plot_curves(history, f"{model_architecture}_{mode}")
    np.save(os.path.join(OUT_DIR, f"cm_{model_architecture}_{mode}.npy"), conf_matrix)
    print(f"TEST {model_architecture}/{mode}: accuracy {test_acc:.4f} | macro-F1 {f1:.4f}")
    return {"model": model_architecture, "input": mode, "val_acc": best_validation, "test_acc": test_acc, "macro_f1": f1}

device: cuda
reading: /kaggle/input/datasets/mdsadiqiqbal/txd-22-a-large-scale-benchmark-dataset/TXD-22 A Large-Scale Benchmark Dataset.csv
rows: 17724 | classes: 6 | chance: 0.167
train/val/test: 12402 2659 2663


In [3]:
x = torch.randn(1000, 1000, device="cuda")
print((x @ x).sum().item())

-572.3671875


In [4]:
tr = train.copy()
tr["tokens"] = tr.a_ids.map(len)
tr["markdown"] = tr.Answer.str.contains(r"(?m)\*\*|^#|^- ", regex=True)
print(tr.groupby("Source").agg(
    mean_tokens=("tokens", "mean"),
    median_tokens=("tokens", "median"),
    pct_over_400=("tokens", lambda s: (s > 400).mean() * 100),
    pct_markdown=("markdown", lambda s: s.mean() * 100)).round(1))

          mean_tokens  median_tokens  pct_over_400  pct_markdown
Source                                                          
ChatGPT         290.5          284.5           6.8           0.0
Claude          342.7          337.0           6.9           0.5
DeepSeek        337.2          337.0          15.3           2.2
Gemini          322.3          355.0          29.4           0.0
Llama           243.5          241.0           0.1           0.0
Qwen            265.8          271.5           0.9           0.0


In [5]:
results = []
for mode in ["output", "input", "both"]:
    for model_architecture in ["cnn", "lstm"]:
        path = os.path.join(OUT_DIR, f"res_{model_architecture}_{mode}.json")
        if os.path.exists(path): #this run already finished before a crash, so reuse it
            r = json.load(open(path))
            print("loaded finished run:", model_architecture, mode)
        else:
            r = run_model(model_architecture, mode)
            json.dump(r, open(path, "w"), default = float) #saved right away
        results.append(r)
output = pd.DataFrame(results)
print("\n", output.round(4).to_string(index = False))
output.to_csv(os.path.join(OUT_DIR, "results.csv"), index = False)
json.dump(classes, open(os.path.join(OUT_DIR, "classes.json"), "w"))


CNN | input type = output | parameters = 5,275,706
 Epoch  |  Train Loss  |  Val Loss  | Val Accuracy |  Elapsed 
------------------------------------------------------------
   1    |   1.411935   |  0.928941  |   65.06   |   4.64   
   2    |   0.941764   |  0.917783  |   62.58   |   3.07   
   3    |   0.805988   |  1.029161  |   63.26   |   3.09   
   4    |   0.718228   |  0.939995  |   64.20   |   3.10   
early stopping
TEST cnn/output: accuracy 0.6433 | macro-F1 0.6088

LSTM | input type = output | parameters = 5,385,734
 Epoch  |  Train Loss  |  Val Loss  | Val Accuracy |  Elapsed 
------------------------------------------------------------
   1    |   1.732687   |  2.028624  |   27.87   |   8.44   
   2    |   1.468282   |  1.507204  |   41.82   |   7.97   
   3    |   1.251226   |  1.080788  |   60.47   |   8.04   
   4    |   0.997538   |  0.895899  |   63.97   |   8.02   
   5    |   0.816092   |  0.857829  |   70.14   |   8.04   
   6    |   0.647343   |  0.653629  |   7

In [6]:
import shutil
shutil.make_archive("/kaggle/working/outputs_6", "zip", "/kaggle/working/outputs_6")

'/kaggle/working/outputs_6.zip'

In [7]:
import numpy as np, json
classes = json.load(open("outputs_6/classes.json"))
for name in ["cnn_output", "lstm_output", "cnn_input", "lstm_input", "cnn_both", "lstm_both"]:
    cm = np.load(f"outputs_6/cm_{name}.npy")
    recall = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
    precision = np.diag(cm) / np.maximum(cm.sum(axis=0), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)
    print("\n==", name, "| accuracy", round(np.trace(cm) / cm.sum(), 4))
    print("true counts     :", cm.sum(axis=1))
    print("predicted counts:", cm.sum(axis=0))
    for i, c in enumerate(classes):
        print(f"  {c:9s} precision {precision[i]:.2f}  recall {recall[i]:.2f}  F1 {f1[i]:.2f}")
    pairs = [(cm[i, j], classes[i], classes[j]) for i in range(6) for j in range(6) if i != j]
    print("  top confusions (true -> predicted):", [f"{t}->{p}: {n}" for n, t, p in sorted(pairs, reverse=True)[:3]])


== cnn_output | accuracy 0.6433
true counts     : [443 443 443 443 448 443]
predicted counts: [307 617 470 497 682  90]
  ChatGPT   precision 0.65  recall 0.45  F1 0.53
  Claude    precision 0.64  recall 0.89  F1 0.75
  DeepSeek  precision 0.65  recall 0.69  F1 0.67
  Gemini    precision 0.63  recall 0.71  F1 0.67
  Llama     precision 0.61  recall 0.94  F1 0.74
  Qwen      precision 0.87  recall 0.18  F1 0.29
  top confusions (true -> predicted): ['Qwen->Llama: 90', 'Qwen->DeepSeek: 88', 'Qwen->ChatGPT: 75']

== lstm_output | accuracy 0.7976
true counts     : [443 443 443 443 448 443]
predicted counts: [522 486 330 364 455 506]
  ChatGPT   precision 0.69  recall 0.81  F1 0.74
  Claude    precision 0.87  recall 0.95  F1 0.91
  DeepSeek  precision 0.85  recall 0.64  F1 0.73
  Gemini    precision 0.84  recall 0.69  F1 0.76
  Llama     precision 0.95  recall 0.96  F1 0.96
  Qwen      precision 0.64  recall 0.73  F1 0.68
  top confusions (true -> predicted): ['DeepSeek->Qwen: 83', 'Qwen->